In [1]:
# ============================================================
# CS3807 - DEEP LEARNING LABORATORY
# EXPERIMENT 4
# Comparative Study of Deep CNN Architectures Using Transfer
# Learning
#
# Models: LeNet-5, AlexNet, VGG16, GoogleNet (InceptionV3),
#         ResNet50
# Dataset: CIFAR-10
#
# ---- NOTES ON THIS VERSION ---------------------------------
# 1. LeNet-5 and AlexNet are NOT available as Keras pretrained
#    models, so they are implemented from scratch, scaled to
#    CIFAR-10 (this is standard practice for this exercise -
#    the originals were sized for MNIST/ImageNet resolutions).
# 2. Keras has no literal "GoogleNet" (Inception-v1) in its
#    model zoo. InceptionV3 (same Inception-module family) is
#    used as the standard substitute - mention this if asked.
# 3. VGG16 and ResNet50 use ImageNet transfer learning, same
#    workflow as the earlier script (freeze base -> train head
#    -> unfreeze last block -> fine-tune).
# 4. To fit 5 architectures inside a reasonable time budget,
#    the dataset is subsampled and epoch counts are kept low.
#    See the CONFIG block below - every knob you'd want to
#    raise (more data, more epochs) is right there.
# 5. At the end, the whole "results" folder is zipped into
#    results.zip so you can download everything in one file.
#
# ALL FIGURES ARE SAVED AS EPS FORMAT @ 600 DPI
# ============================================================

import os
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50, InceptionV3
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 0. CONFIG - adjust these if you have more time / a GPU
# ============================================================

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

CLASS_NAMES = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]
NUM_CLASSES = 10

# Subsample sizes (raise these if you have time/GPU)
TRAIN_SUBSET = 5000
VAL_SUBSET = 1200
TEST_SUBSET = 1500

BATCH_SIZE = 64

# Per-architecture settings: image size, epochs, fine-tune epochs
# (from-scratch models get more epochs since they start from
# random weights; transfer-learning models get fewer since the
# conv base is already trained)
CONFIG = {
    "LeNet-5":   {"image_size": 32, "epochs": 15, "fine_tune_epochs": 0},
    "AlexNet":   {"image_size": 64, "epochs": 3, "fine_tune_epochs": 0},
    "VGG16":     {"image_size": 96, "epochs": 3,  "fine_tune_epochs": 2},
    "GoogleNet": {"image_size": 96, "epochs": 3,  "fine_tune_epochs": 2},  # InceptionV3
    "ResNet50":  {"image_size": 96, "epochs": 3,  "fine_tune_epochs": 2},
}

LEARNING_RATE = 0.001
FINE_TUNE_LR = 0.00001

print("=" * 70)
print("CS3807 - DEEP LEARNING LABORATORY")
print("EXPERIMENT 4 - CNN ARCHITECTURE COMPARISON (LeNet-5, AlexNet,")
print("VGG16, GoogleNet/InceptionV3, ResNet50)")
print("=" * 70)

overall_start = time.time()


# ============================================================
# 1. LOAD + NORMALIZE + SUBSAMPLE CIFAR-10 (shared across all models)
# ============================================================

print("\nLoading CIFAR-10 dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

x_val_full = x_train[-5000:]
y_val_full = y_train_cat[-5000:]
x_train_full = x_train[:-5000]
y_train_full = y_train_cat[:-5000]

rng = np.random.RandomState(SEED)
train_idx = rng.choice(len(x_train_full), size=TRAIN_SUBSET, replace=False)
val_idx = rng.choice(len(x_val_full), size=VAL_SUBSET, replace=False)
test_idx = rng.choice(len(x_test), size=TEST_SUBSET, replace=False)

x_train_final = x_train_full[train_idx]
y_train_final = y_train_full[train_idx]
x_val = x_val_full[val_idx]
y_val = y_val_full[val_idx]
x_test_sub = x_test[test_idx]
y_test_sub = y_test[test_idx]
y_test_cat_sub = y_test_cat[test_idx]

print("\nDataset Split (subsampled for speed)")
print("-" * 50)
print("Training   :", x_train_final.shape)
print("Validation :", x_val.shape)
print("Testing    :", x_test_sub.shape)


# Sample images figure (once, shared)
plt.figure(figsize=(12, 7))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[int(y_train[i][0])], fontsize=10)
    plt.axis("off")
plt.suptitle("Sample CIFAR-10 Images", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "00_sample_cifar10.eps"),
            format="eps", dpi=600, bbox_inches="tight")
plt.close()
print("Saved: 00_sample_cifar10.eps")


# ============================================================
# 2. HELPERS
# ============================================================

def make_datasets(image_size, preprocess_fn, no_resize=False):
    """Build cached, batched tf.data pipelines at the given image size."""

    def prep(images, labels):
        if not no_resize:
            images = tf.image.resize(images, (image_size, image_size))
            images = images * 255.0
        else:
            images = images * 255.0
        images = preprocess_fn(images)
        return images, labels

    train_ds = (
        tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final))
        .map(prep, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .shuffle(buffer_size=TRAIN_SUBSET, seed=SEED)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((x_val, y_val))
        .map(prep, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    test_ds = (
        tf.data.Dataset.from_tensor_slices((x_test_sub, y_test_cat_sub))
        .map(prep, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    return train_ds, val_ds, test_ds


def plot_curves(history_dict, model_name, tag):
    """Save 2 EPS plots per model: accuracy (train+val), loss (train+val)."""
    epochs_range = range(1, len(history_dict["accuracy"]) + 1)

    plt.figure(figsize=(8, 6))
    plt.plot(epochs_range, history_dict["accuracy"], marker="o", linewidth=2, label="Training Accuracy")
    plt.plot(epochs_range, history_dict["val_accuracy"], marker="s", linewidth=2, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    fname_acc = f"{tag}_accuracy.eps"
    plt.savefig(os.path.join(RESULT_DIR, fname_acc), format="eps", dpi=600, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.plot(epochs_range, history_dict["loss"], marker="o", linewidth=2, label="Training Loss")
    plt.plot(epochs_range, history_dict["val_loss"], marker="s", linewidth=2, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    fname_loss = f"{tag}_loss.eps"
    plt.savefig(os.path.join(RESULT_DIR, fname_loss), format="eps", dpi=600, bbox_inches="tight")
    plt.close()

    print(f"Saved: {fname_acc}, {fname_loss}")


def evaluate_model(model, test_ds, y_true_labels):
    y_true = []
    y_pred = []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return acc, prec, rec, f1, y_true, y_pred


def unfreeze_by_prefix(base_model, prefixes):
    """Freeze everything, then unfreeze layers whose name starts with
    any of the given prefixes (used for fine-tuning the last block)."""
    base_model.trainable = True
    for layer in base_model.layers:
        layer.trainable = any(layer.name.startswith(p) for p in prefixes)


# Results collector
results = []


# ============================================================
# 3. LeNet-5 (from scratch, native 32x32 resolution)
# ============================================================

print("\n" + "=" * 70)
print("MODEL 1/5: LeNet-5 (from scratch)")
print("=" * 70)

cfg = CONFIG["LeNet-5"]
train_ds, val_ds, test_ds = make_datasets(
    image_size=32,
    preprocess_fn=lambda x: x / 255.0,  # simple [0,1] normalization
    no_resize=True
)

lenet5 = models.Sequential([
    layers.Conv2D(6, (5, 5), activation="tanh", padding="same", input_shape=(32, 32, 3)),
    layers.AveragePooling2D(2),
    layers.Conv2D(16, (5, 5), activation="tanh"),
    layers.AveragePooling2D(2),
    layers.Flatten(),
    layers.Dense(120, activation="tanh"),
    layers.Dense(84, activation="tanh"),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="LeNet5")

lenet5.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
               loss="categorical_crossentropy", metrics=["accuracy"])
lenet5.summary()

t0 = time.time()
hist = lenet5.fit(train_ds, validation_data=val_ds, epochs=cfg["epochs"])
train_time = time.time() - t0

plot_curves(hist.history, "LeNet-5", "01_lenet5")

acc, prec, rec, f1, _, _ = evaluate_model(lenet5, test_ds, y_test_sub)
results.append({
    "Model": "LeNet-5", "Parameters": lenet5.count_params(),
    "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
    "Training Time (s)": train_time
})
print(f"LeNet-5 -> Accuracy: {acc:.4f}, Time: {train_time:.2f}s")


# ============================================================
# 4. AlexNet (from scratch, resized to 64x64)
# ============================================================

print("\n" + "=" * 70)
print("MODEL 2/5: AlexNet (from scratch, scaled for CIFAR-10)")
print("=" * 70)

cfg = CONFIG["AlexNet"]
train_ds, val_ds, test_ds = make_datasets(
    image_size=cfg["image_size"],
    preprocess_fn=lambda x: x / 255.0
)

alexnet = models.Sequential([
    layers.Conv2D(96, (5, 5), strides=1, activation="relu", padding="same",
                  input_shape=(cfg["image_size"], cfg["image_size"], 3)),
    layers.MaxPooling2D(2),
    layers.Conv2D(256, (5, 5), activation="relu", padding="same"),
    layers.MaxPooling2D(2),
    layers.Conv2D(384, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(384, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(1024, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1024, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="AlexNet")

alexnet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                 loss="categorical_crossentropy", metrics=["accuracy"])
alexnet.summary()

t0 = time.time()
hist = alexnet.fit(train_ds, validation_data=val_ds, epochs=cfg["epochs"])
train_time = time.time() - t0

plot_curves(hist.history, "AlexNet", "02_alexnet")

acc, prec, rec, f1, _, _ = evaluate_model(alexnet, test_ds, y_test_sub)
results.append({
    "Model": "AlexNet", "Parameters": alexnet.count_params(),
    "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
    "Training Time (s)": train_time
})
print(f"AlexNet -> Accuracy: {acc:.4f}, Time: {train_time:.2f}s")


# ============================================================
# 5. VGG16 (ImageNet transfer learning + fine-tuning)
# ============================================================

print("\n" + "=" * 70)
print("MODEL 3/5: VGG16 (transfer learning)")
print("=" * 70)

cfg = CONFIG["VGG16"]
train_ds, val_ds, test_ds = make_datasets(cfg["image_size"], vgg_preprocess)

vgg_base = VGG16(weights="imagenet", include_top=False,
                  input_shape=(cfg["image_size"], cfg["image_size"], 3))
vgg_base.trainable = False

vgg16_model = models.Sequential([
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="VGG16_Transfer")

vgg16_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                     loss="categorical_crossentropy", metrics=["accuracy"])
vgg16_model.summary()

t0 = time.time()
hist1 = vgg16_model.fit(train_ds, validation_data=val_ds, epochs=cfg["epochs"])

# Fine-tune: unfreeze block5
unfreeze_by_prefix(vgg_base, ["block5"])
vgg16_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
                     loss="categorical_crossentropy", metrics=["accuracy"])
hist2 = vgg16_model.fit(train_ds, validation_data=val_ds, epochs=cfg["fine_tune_epochs"])
train_time = time.time() - t0

combined_hist = {
    "accuracy": hist1.history["accuracy"] + hist2.history["accuracy"],
    "val_accuracy": hist1.history["val_accuracy"] + hist2.history["val_accuracy"],
    "loss": hist1.history["loss"] + hist2.history["loss"],
    "val_loss": hist1.history["val_loss"] + hist2.history["val_loss"],
}
plot_curves(combined_hist, "VGG16", "03_vgg16")

acc, prec, rec, f1, y_true_vgg, y_pred_vgg = evaluate_model(vgg16_model, test_ds, y_test_sub)
results.append({
    "Model": "VGG16", "Parameters": vgg16_model.count_params(),
    "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
    "Training Time (s)": train_time
})
print(f"VGG16 -> Accuracy: {acc:.4f}, Time: {train_time:.2f}s")

# VGG16 is treated as the "primary" detailed model: confusion matrix +
# classification report (matches Task 5 of the lab manual).
report = classification_report(y_true_vgg, y_pred_vgg, target_names=CLASS_NAMES,
                                digits=4, zero_division=0)
with open(os.path.join(RESULT_DIR, "vgg16_classification_report.txt"), "w") as f:
    f.write(report)
print("\nVGG16 Classification Report:\n", report)

cm = confusion_matrix(y_true_vgg, y_pred_vgg)
plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix - VGG16")
plt.colorbar()
tick_marks = np.arange(NUM_CLASSES)
plt.xticks(tick_marks, CLASS_NAMES, rotation=45, ha="right")
plt.yticks(tick_marks, CLASS_NAMES)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
threshold = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, format(cm[i, j], "d"), horizontalalignment="center",
                  color="white" if cm[i, j] > threshold else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "03_vgg16_confusion_matrix.eps"),
            format="eps", dpi=600, bbox_inches="tight")
plt.close()
print("Saved: 03_vgg16_confusion_matrix.eps")


# ============================================================
# 6. GoogleNet substitute: InceptionV3 (ImageNet transfer learning)
# ============================================================

print("\n" + "=" * 70)
print("MODEL 4/5: GoogleNet (InceptionV3, transfer learning)")
print("Note: Keras has no literal Inception-v1/GoogleNet weights;")
print("InceptionV3 (same Inception-module family) is used instead.")
print("=" * 70)

cfg = CONFIG["GoogleNet"]
# InceptionV3 requires input >= 75x75; 96 satisfies that.
train_ds, val_ds, test_ds = make_datasets(cfg["image_size"], inception_preprocess)

inception_base = InceptionV3(weights="imagenet", include_top=False,
                              input_shape=(cfg["image_size"], cfg["image_size"], 3))
inception_base.trainable = False

googlenet_model = models.Sequential([
    inception_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="GoogleNet_InceptionV3_Transfer")

googlenet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                         loss="categorical_crossentropy", metrics=["accuracy"])
googlenet_model.summary()

t0 = time.time()
hist1 = googlenet_model.fit(train_ds, validation_data=val_ds, epochs=cfg["epochs"])

# Fine-tune: unfreeze the last Inception block ("mixed10")
unfreeze_by_prefix(inception_base, ["mixed10", "mixed9"])
googlenet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
                         loss="categorical_crossentropy", metrics=["accuracy"])
hist2 = googlenet_model.fit(train_ds, validation_data=val_ds, epochs=cfg["fine_tune_epochs"])
train_time = time.time() - t0

combined_hist = {
    "accuracy": hist1.history["accuracy"] + hist2.history["accuracy"],
    "val_accuracy": hist1.history["val_accuracy"] + hist2.history["val_accuracy"],
    "loss": hist1.history["loss"] + hist2.history["loss"],
    "val_loss": hist1.history["val_loss"] + hist2.history["val_loss"],
}
plot_curves(combined_hist, "GoogleNet (InceptionV3)", "04_googlenet")

acc, prec, rec, f1, _, _ = evaluate_model(googlenet_model, test_ds, y_test_sub)
results.append({
    "Model": "GoogleNet (InceptionV3)", "Parameters": googlenet_model.count_params(),
    "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
    "Training Time (s)": train_time
})
print(f"GoogleNet -> Accuracy: {acc:.4f}, Time: {train_time:.2f}s")


# ============================================================
# 7. ResNet50 (ImageNet transfer learning + fine-tuning)
# ============================================================

print("\n" + "=" * 70)
print("MODEL 5/5: ResNet50 (transfer learning)")
print("=" * 70)

cfg = CONFIG["ResNet50"]
train_ds, val_ds, test_ds = make_datasets(cfg["image_size"], resnet_preprocess)

resnet_base = ResNet50(weights="imagenet", include_top=False,
                        input_shape=(cfg["image_size"], cfg["image_size"], 3))
resnet_base.trainable = False

resnet50_model = models.Sequential([
    resnet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="ResNet50_Transfer")

resnet50_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                        loss="categorical_crossentropy", metrics=["accuracy"])
resnet50_model.summary()

t0 = time.time()
hist1 = resnet50_model.fit(train_ds, validation_data=val_ds, epochs=cfg["epochs"])

# Fine-tune: unfreeze the last residual block ("conv5_block")
unfreeze_by_prefix(resnet_base, ["conv5_block"])
resnet50_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
                        loss="categorical_crossentropy", metrics=["accuracy"])
hist2 = resnet50_model.fit(train_ds, validation_data=val_ds, epochs=cfg["fine_tune_epochs"])
train_time = time.time() - t0

combined_hist = {
    "accuracy": hist1.history["accuracy"] + hist2.history["accuracy"],
    "val_accuracy": hist1.history["val_accuracy"] + hist2.history["val_accuracy"],
    "loss": hist1.history["loss"] + hist2.history["loss"],
    "val_loss": hist1.history["val_loss"] + hist2.history["val_loss"],
}
plot_curves(combined_hist, "ResNet50", "05_resnet50")

acc, prec, rec, f1, _, _ = evaluate_model(resnet50_model, test_ds, y_test_sub)
results.append({
    "Model": "ResNet50", "Parameters": resnet50_model.count_params(),
    "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
    "Training Time (s)": train_time
})
print(f"ResNet50 -> Accuracy: {acc:.4f}, Time: {train_time:.2f}s")


# ============================================================
# 8. COMPARISON TABLE
# ============================================================

results_df = pd.DataFrame(results)
results_df["Accuracy (%)"] = results_df["Accuracy"] * 100

print("\n" + "=" * 70)
print("CNN ARCHITECTURE COMPARISON - FINAL RESULTS")
print("=" * 70)
print(results_df.to_string(index=False))

results_df.to_csv(os.path.join(RESULT_DIR, "model_comparison.csv"), index=False)


# ============================================================
# 9. COMPARISON BAR CHARTS (EPS)
# ============================================================

plt.figure(figsize=(9, 6))
plt.bar(results_df["Model"], results_df["Accuracy (%)"], color="steelblue")
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy Comparison Across Architectures")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "06_accuracy_comparison.eps"),
            format="eps", dpi=600, bbox_inches="tight")
plt.close()
print("Saved: 06_accuracy_comparison.eps")

plt.figure(figsize=(9, 6))
plt.bar(results_df["Model"], results_df["Training Time (s)"], color="indianred")
plt.ylabel("Training Time (seconds)")
plt.title("Training Time Comparison Across Architectures")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "07_training_time_comparison.eps"),
            format="eps", dpi=600, bbox_inches="tight")
plt.close()
print("Saved: 07_training_time_comparison.eps")


# ============================================================
# 10. ZIP RESULTS FOLDER FOR DOWNLOAD
# ============================================================

zip_path = shutil.make_archive("results", "zip", RESULT_DIR)
print(f"\nAll results zipped to: {zip_path}")
print("On Google Colab, download it with:")
print("  from google.colab import files")
print("  files.download('results.zip')")
print("On a local Jupyter/VS Code setup, the zip is in your working directory -")
print("just locate it in the file browser and copy/drag it to your laptop.")


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

overall_time = time.time() - overall_start

print("\n" + "=" * 70)
print("EXPERIMENT COMPLETED - ALL 5 ARCHITECTURES")
print("=" * 70)
print(f"\nTotal wall-clock time: {overall_time/60:.1f} minutes")
print("\nGenerated files in 'results/':")
for f in sorted(os.listdir(RESULT_DIR)):
    print("  ->", f)
print("\nDone!")

CS3807 - DEEP LEARNING LABORATORY
EXPERIMENT 4 - CNN ARCHITECTURE COMPARISON (LeNet-5, AlexNet,
VGG16, GoogleNet/InceptionV3, ResNet50)

Loading CIFAR-10 dataset...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2120s 12us/step

Dataset Split (subsampled for speed)
--------------------------------------------------
Training   : (5000, 32, 32, 3)
Validation : (1200, 32, 32, 3)
Testing    : (1500, 32, 32, 3)
Saved: 00_sample_cifar10.eps

MODEL 1/5: LeNet-5 (from scratch)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "LeNet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 6)      │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 16, 16, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 12, 12, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 6, 6, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        69,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,126 (324.71 KB)

...

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/3
79/79 ━━━━━━━━━━━━━━━━━━━━ 129s 2s/step - accuracy: 0.7370 - loss: 0.8557 - val_accuracy: 0.7958 - val_loss: 0.6164
Epoch 2/3
79/79 ━━━━━━━━━━━━━━━━━━━━ 120s 2s/step - accuracy: 0.8790 - loss: 0.3613 - val_accuracy: 0.8075 - val_loss: 0.5856
Epoch 3/3
79/79 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.9224 - loss: 0.2251 - val_accuracy: 0.8108 - val_loss: 0.6066
Epoch 1/2
79/79 ━━━━━━━━━━━━━━━━━━━━ 306s 4s/step - accuracy: 0.8638 - loss: 0.4404 - val_accuracy: 0.8150 - val_loss: 0.5990
Epoch 2/2
79/79 ━━━━━━━━━━━━━━━━━━━━ 313s 4s/step - accuracy: 0.9550 - loss: 0.2311 - val_accuracy: 0.8225 - val_loss: 0.5817


Saved: 05_resnet50_accuracy.eps, 05_resnet50_loss.eps


ResNet50 -> Accuracy: 0.8133, Time: 989.02s

CNN ARCHITECTURE COMPARISON - FINAL RESULTS
                  Model  Parameters  Accuracy  Precision   Recall  F1-score  Training Time (s)  Accuracy (%)
                LeNet-5       83126  0.422000   0.431378 0.422000  0.416243          50.819046     42.200000
                AlexNet    21557642  0.309333   0.389521 0.309333  0.283656        1985.298511     30.933333
                  VGG16    14781642  0.776000   0.776852 0.776000  0.775370        2412.282125     77.600000
GoogleNet (InceptionV3)    22066346  0.651333   0.651741 0.651333  0.650824         369.053100     65.133333
               ResNet50    23851274  0.813333   0.814056 0.813333  0.813355         989.021286     81.333333
Saved: 06_accuracy_comparison.eps


Saved: 07_training_time_comparison.eps

All results zipped to: /content/results.zip
On Google Colab, download it with:
  from google.colab import files
  files.download('results.zip')
On a local Jupyter/VS Code setup, the zip is in your working directory -
just locate it in the file browser and copy/drag it to your laptop.

EXPERIMENT COMPLETED - ALL 5 ARCHITECTURES

Total wall-clock time: 136.9 minutes

Generated files in 'results/':
  -> 00_sample_cifar10.eps
  -> 01_lenet5_accuracy.eps
  -> 01_lenet5_loss.eps
  -> 02_alexnet_accuracy.eps
  -> 02_alexnet_loss.eps
  -> 03_vgg16_accuracy.eps
  -> 03_vgg16_confusion_matrix.eps
  -> 03_vgg16_loss.eps
  -> 04_googlenet_accuracy.eps
  -> 04_googlenet_loss.eps
  -> 05_resnet50_accuracy.eps
  -> 05_resnet50_loss.eps
  -> 06_accuracy_comparison.eps
  -> 07_training_time_comparison.eps
  -> model_comparison.csv
  -> vgg16_classification_report.txt

Done!
